# Practice Lab 3.5 - Prompt Caching, Batching, Cost Engineering (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 3 - Lesson 3.5 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-3.5.html` (6 exercises).
Each exercise below repeats the task and success criteria, then shows a runnable worked
solution. Run it top to bottom.

- The cache exercises read `cache_creation_input_tokens` / `cache_read_input_tokens` off
  `response.usage`. **No key needed to run:** the keyless guard in the next cell mocks the
  anthropic + openai clients so every cell executes end to end. Keyless usage is zero, so the
  printed dollar deltas are degenerate - set `ANTHROPIC_API_KEY` (and `OPENAI_API_KEY` for the
  Ex 4 batch) to reproduce the real ~79% cache saving and the 50% batch discount.
- Production context: a regional-model vendor lending advisor + Stripe merchant-support RAG,
  code-mixed queries, USD costs on the Aug 2026 Sonnet 5 rate sheet, synthetic policy + IDs only.
- Cost with a live key: well under the $25 lesson cap (small Sonnet / Haiku calls; the Ex 4 batch is one job).


In [ ]:
!pip install --quiet anthropic openai prometheus-client


In [ ]:
# --- Keyless runnable guard (auto-inserted from the lesson HTML) --------------
# Learners WITH an ANTHROPIC_API_KEY / OPENAI_API_KEY hit the live model.
# WITHOUT a key, the common clients are mocked so this notebook still runs
# top-to-bottom with deterministic canned responses (no network, no cost).
import os
_HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
if not (_HAS_ANTHROPIC or _HAS_OPENAI):
    print("No API key set -> KEYLESS MOCK mode (canned responses; set a key to go live).")
    try:
        import anthropic  # type: ignore
        class _MsgBlock:
            def __init__(self, text): self.text = text; self.type = "text"
        class _Msg:
            def __init__(self, text):
                self.content = [_MsgBlock(text)]
                self.stop_reason = "end_turn"
                self.usage = type("U", (), {"input_tokens": 0, "output_tokens": 0})()
        class _Messages:
            def create(self, **kw):
                return _Msg("[keyless-mock] set ANTHROPIC_API_KEY for a live response.")
        class _Anthropic:
            def __init__(self, *a, **k): self.messages = _Messages()
        anthropic.Anthropic = _Anthropic
        if hasattr(anthropic, "AsyncAnthropic"): anthropic.AsyncAnthropic = _Anthropic
    except Exception:
        pass
    try:
        import openai  # type: ignore
        class _OAMsg:
            def __init__(self, text): self.content = text
        class _OAChoice:
            def __init__(self, text): self.message = _OAMsg(text)
        class _OAResp:
            def __init__(self, text): self.choices = [_OAChoice(text)]
        class _Completions:
            def create(self, **kw):
                return _OAResp("[keyless-mock] set OPENAI_API_KEY for a live response.")
        class _Chat:
            def __init__(self): self.completions = _Completions()
        class _OpenAI:
            def __init__(self, *a, **k): self.chat = _Chat()
        openai.OpenAI = _OpenAI
        if hasattr(openai, "AsyncOpenAI"): openai.AsyncOpenAI = _OpenAI
    except Exception:
        pass
# ----------------------------------------------------------------------------


---
## Exercise 1 - Add cache_control and verify cache_hit metadata

**Task.** Build a system prompt of ~1200 tokens (the regional-model vendor-style lending advisor preamble below). Add `cache_control: {"type":"ephemeral"}` on the system block. Call the model twice in quick succession with two different user queries. Print the two `usage` objects. Confirm: call 1 shows `cache_creation_input_tokens > 0` and `cache_read_input_tokens == 0`; call 2 shows the inverse.

**Success criteria**
- System block uses the structured form (`[{"type":"text","text":...,"cache_control":...}]`), not a bare string
- Call 1's `cache_creation_input_tokens` is > 1024 (below that, Sonnet 5 silently skips caching)
- Call 2's `cache_read_input_tokens` equals call 1's `cache_creation_input_tokens`
- Both user queries differ - the cache key is the system block, not the user message
- Total spend under $0.50 (2 short Sonnet calls)


In [ ]:
import anthropic
client = anthropic.Anthropic()

# Pad the system prompt to comfortably exceed Sonnet 5's 1024-token cache minimum.
SYSTEM_PREAMBLE = ("""You are a regional-model vendor lending advisor for regional-market small businesses.
Follow OCC / FCA master directions on lending. Always declare a GDPR purpose for any
PII access. Respond in the user's language (code-mixed if the query mixes languages,
Spanish if Spanish, English if English). Never quote a loan amount you weren't asked about.

POLICY EXTRACTS (paraphrased, illustrative; not legal advice):
- KYC must be revalidated every 12 months.
- Open Banking consent is required before pulling bank statements.
- Credit score below 650 routes to manual underwriting.
- All SSN / national ID strings must be masked before being echoed back.

FEW-SHOT EXAMPLES:
Q: 'Hola, necesito un loan de $6,000, mi salary es 60K.'
A: '$6,000 with 60K salary - monthly payment roughly $130/month over 5 years at 11% reducing.
   Recommended: process via lookup_credit first.'

Q: 'Mi credit score es 720, puedo get a home loan?'
A: 'A credit score of 720 is healthy. Home loan eligibility depends on income, tenure, and LTV.'
""" + "Filler stable context. " * 80).strip()

def call(user_query: str):
    return client.messages.create(
        model="claude-sonnet-5",
        max_tokens=200,
        system=[{
            "type": "text",
            "text": SYSTEM_PREAMBLE,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": user_query}],
    )

r1 = call("Mi credit score es 700, necesito $3,600 for 2 years - eligible?")
r2 = call("Calcula el monthly payment del personal loan: $2,400, 18%, 24 months.")

for i, r in enumerate([r1, r2], 1):
    u = r.usage
    print(f"call {i}: input={u.input_tokens}  "
          f"cache_create={getattr(u,'cache_creation_input_tokens',0)}  "
          f"cache_read={getattr(u,'cache_read_input_tokens',0)}")
# Live key: call 1 shows cache_create > 1024 with cache_read == 0; call 2 the inverse.
# Keyless mock returns zero-usage, so both rows print zeros - set ANTHROPIC_API_KEY for the real counters.


---
## Exercise 2 - Measure $cost: 10 cached vs 10 uncached calls

**Task.** Run the same code-mixed query through the regional-model vendor advisor 10 times with caching off, then 10 times with caching on. Compute the total $spent in each mode using the Aug 2026 Sonnet 5 rate sheet (hedged below). Print a 3-row table: uncached $, cached $, savings %.

**Success criteria**
- Rate constants hedged in a comment ("Aug 2026, check on call day")
- The cost function correctly subtracts cached tokens from the full-price line (do not double-count)
- Savings >= 60% on a 10-call run (the prefix dominates; if you see less, the prefix is too small)
- The first cached call shows a non-zero `cache_creation`; the next 9 show non-zero `cache_read`
- Total spend under $5


In [ ]:
import anthropic
client = anthropic.Anthropic()

# Sonnet 5 Aug 2026 rate sheet (hedged - check Anthropic's pricing page on call day).
RATE_IN  = 2.0 / 1_000_000   # $per input token
RATE_OUT = 10.0 / 1_000_000   # $per output token
RATE_CACHE_WRITE = RATE_IN * 1.25
RATE_CACHE_READ  = RATE_IN * 0.10

SYSTEM = ("a regional-model vendor lending advisor. Follow OCC / FCA guidance. " * 60).strip()  # ~1200 tokens
QUERY = "Monthly payment for $2,400 at 18% over 24 months?"

def cost(u, cached: bool) -> float:
    # Cached tokens are subtracted from the full-price line and re-priced at the write/read rate,
    # so we never double-count the prefix.
    cw = getattr(u, "cache_creation_input_tokens", 0) or 0
    cr = getattr(u, "cache_read_input_tokens", 0) or 0
    base_in = u.input_tokens  # NOT including cached portion
    return (base_in * RATE_IN + cw * RATE_CACHE_WRITE
            + cr * RATE_CACHE_READ + u.output_tokens * RATE_OUT)

def run(cached: bool, n: int = 10) -> float:
    total = 0.0
    sys_block = ([{"type":"text","text":SYSTEM,"cache_control":{"type":"ephemeral"}}]
                 if cached else SYSTEM)
    for _ in range(n):
        r = client.messages.create(model="claude-sonnet-5", max_tokens=120,
                                   system=sys_block,
                                   messages=[{"role":"user","content":QUERY}])
        total += cost(r.usage, cached)
    return total

uncached = run(cached=False)
cached   = run(cached=True)
saving = (1 - cached / uncached) * 100 if uncached else 0.0  # guard keyless-mock zero-usage

print(f"{'mode':10s} | $total")
print("-" * 26)
print(f"{'uncached':10s} | {uncached:.3f}")
print(f"{'cached':10s} | {cached:.3f}")
print(f"\nSaving: {saving:.1f}%")
# Live key: uncached pays full price 10x; cached pays 1.25x once + 0.10x nine times -> ~79% off
# the prefix. Keyless mock returns zero-usage, so both totals are 0.00 and Saving prints 0.0%.


---
## Exercise 3 - Optimize a real RAG prompt - move stable above the breakpoint

**Task.** Refactor the provided 4-section Meridian Capital RAG prompt. Identify which sections are stable across the day (system role, policy excerpts, fixed few-shot examples) and which are per-call (the user query, the retrieved-for-this-query chunks, the timestamp). Build the request with the stable sections as content blocks above the breakpoint and the volatile sections below it. Verify with 5 calls that `cache_read_input_tokens` is non-zero on calls 2-5.

**Success criteria**
- System is a list of 4 content blocks, cache_control on the LAST block only
- Volatile fields (timestamp, retrieved chunks, user query) all live in the `messages` payload, not the system list
- Call 1 shows `cache_creation > 0`; calls 2-5 show `cache_read > 0` equal to call 1's creation count
- The cache_read counter is identical across calls 2-5 (proves the stable section is truly stable)
- Total spend under $2


In [ ]:
import anthropic, datetime
client = anthropic.Anthropic()

# STABLE - same for every call until the policy revision lands (~weekly):
SYSTEM_ROLE = "You are a Meridian Capital lending advisor. Cite policy where relevant."
OCC_POLICY  = ("OCC / FCA master directions excerpt (paraphrased). " * 50).strip()
GDPR_POLICY = ("GDPR gazette excerpt (paraphrased). " * 40).strip()
FEW_SHOTS   = ("Few-shot Q/A pairs spanning code-mixed / Spanish / English. " * 30).strip()

# VOLATILE - changes per call:
def per_call_block(user_q: str, retrieved_chunks: list[str]) -> str:
    today = datetime.date.today().isoformat()
    return (f"TODAY: {today}\n"
            f"RETRIEVED:\n" + "\n---\n".join(retrieved_chunks)
            + f"\n\nUSER QUERY: {user_q}")

def ask(user_q: str, chunks: list[str]):
    return client.messages.create(
        model="claude-sonnet-5",
        max_tokens=300,
        system=[
            {"type":"text","text":SYSTEM_ROLE},
            {"type":"text","text":OCC_POLICY},
            {"type":"text","text":GDPR_POLICY},
            # Breakpoint on the LAST stable block - everything above is cached together:
            {"type":"text","text":FEW_SHOTS,"cache_control":{"type":"ephemeral"}},
        ],
        messages=[{"role":"user","content":per_call_block(user_q, chunks)}],
    )

calls = [
    ("Mi credit score es 680, $6,000 business loan - eligible?",
     ["Chunk A: credit score 650+ qualifies under standard SMB.", "Chunk B: $6,000 tenor up to 36 months."]),
    ("Calcula el monthly payment del personal loan: $3,600, 14%, 36 months.",
     ["Chunk C: monthly payment formula = P*r*(1+r)^n / ((1+r)^n - 1)."]),
    ("Como aplico para un home loan top-up?", ["Chunk D: Top-up needs existing loan in good standing 12+ months."]),
    ("Para un secured loan, revisan el credit score?", ["Chunk E: Secured loans are asset-backed; credit score is checked but not gating."]),
    ("Cual es el script para insurance cross-sell?", ["Chunk F: insurance regulators guidance: cross-sell only post-consent."]),
]

for i, (q, ch) in enumerate(calls, 1):
    r = ask(q, ch)
    u = r.usage
    print(f"call {i}: in={u.input_tokens:<4} "
          f"create={getattr(u,'cache_creation_input_tokens',0):<5} "
          f"read={getattr(u,'cache_read_input_tokens',0)}")
# Live key: call 1 writes the 4-block stable prefix; calls 2-5 read it (identical read count each).
# Keyless mock returns zero-usage, so every column prints 0 - set ANTHROPIC_API_KEY for the real shape.


---
## Exercise 4 - Submit a 100-query code-mixed batch job and verify the 50% discount

**Task.** Take 100 code-mixed lending-intent queries. Submit them via OpenAI's batch API as a single JSONL job. Poll for completion. Download the results. Compute the realised $/query and compare to the realtime rate. Verify the discount lands in the 45-55% range (50% nominal with rounding).

**Success criteria**
- JSONL has exactly 100 lines; every `custom_id` is unique
- Batch reaches `completed` within the 24h SLA (in practice 30-120 min)
- Realised batch cost is 45-55% of the equivalent realtime cost
- Result file is parsed; intent label appears in every `response.body.choices[0].message.content`
- Total spend under $3 (the batch run itself + a small realtime smoke test)


In [ ]:
import json, time, io
from openai import OpenAI
client = OpenAI()

# Aug 2026 rate sheet for gpt-5.6-luna (hedged; check on call day).
RATE_IN_RT  = 0.20 / 1_000_000   # $per input token, realtime
RATE_OUT_RT = 1.20 / 1_000_000

CODEMIXED = [
    "Necesito un loan de $2,400, mi salary es 45K. Eligible?",
    "Calcula el monthly payment: $3,600, 13%, 24 months.",
    "Mi credit score es 690, puedo get a home loan?",
    "Personal loan vs business loan - cual es better for a shop?",
    "Cuanto dan por un secured loan?",
]
while len(CODEMIXED) < 100:
    CODEMIXED.append(CODEMIXED[len(CODEMIXED) % 5] + f" (variant {len(CODEMIXED)})")

# 1. Build the JSONL batch input - one complete JSON object per line, unique custom_ids.
lines = []
for i, q in enumerate(CODEMIXED):
    lines.append(json.dumps({
        "custom_id": f"q-{i:03d}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-5.6-luna",
            "messages": [
                {"role":"system","content":"Classify the lending intent in one word: payment, eligibility, comparison, other."},
                {"role":"user","content":q},
            ],
            "max_tokens": 8,
        },
    }))
payload = "\n".join(lines).encode()
assert len(lines) == 100
assert len({json.loads(l)["custom_id"] for l in lines}) == 100
print(f"built {len(lines)}-line JSONL, all custom_ids unique")

# 2-4. Upload, submit, poll, pull results, compute the realised $. The files/batches
# endpoints need a live OpenAI key; guard so the notebook still runs keyless.
try:
    f = client.files.create(file=io.BytesIO(payload), purpose="batch")
    b = client.batches.create(input_file_id=f.id, endpoint="/v1/chat/completions",
                              completion_window="24h")
    print(f"batch id: {b.id} status: {b.status}")
    while b.status not in ("completed", "failed", "cancelled", "expired"):
        time.sleep(15)
        b = client.batches.retrieve(b.id)
    print(f"final status: {b.status}")
    out = client.files.content(b.output_file_id).read().decode()
    in_tok = out_tok = 0
    for ln in out.strip().split("\n"):
        j = json.loads(ln)
        u = j["response"]["body"]["usage"]
        in_tok += u["prompt_tokens"]
        out_tok += u["completion_tokens"]
    cost_realtime = in_tok * RATE_IN_RT + out_tok * RATE_OUT_RT
    cost_batch = cost_realtime * 0.5
    saving = (1 - cost_batch / cost_realtime) * 100 if cost_realtime else 0.0
    print(f"realtime would cost ${cost_realtime:.4f} | batch cost ${cost_batch:.4f} | saving {saving:.0f}%")
except Exception as e:
    print(f"[keyless/offline] batch submit skipped ({type(e).__name__}). "
          f"The 100-line JSONL above is the correct submit shape; set OPENAI_API_KEY to run it live.")
    print("Live run: batch bills at ~50% of realtime with a 24h SLA (usually 30-120 min actual).")


---
## Exercise 5 - Cost-dashboard module: Prometheus exporter + Grafana panel

**Task.** Write a `cost_exporter.py` module that wraps the Anthropic client. On every call it increments two Prometheus metrics: `gen_ai_cost_usd_total` (Counter, labels: customer, tier, model) and `gen_ai_cache_hit_ratio` (Gauge, labels: customer, tier, model). Expose them on `:9100/metrics`. Then write `dashboard.json` - a minimal Grafana panel spec showing $/customer/day and cache-hit-rate on the same time-series. Smoke-test by hitting `:9100/metrics` and confirming both metrics surface.

**Success criteria**
- `cost_exporter.py` exposes both metrics on :9100/metrics, scrapeable by Prometheus
- Labels include `customer`, `tier`, `model` at minimum (GDPR audit-trail evidence)
- The cost calculation includes the cache_write 1.25x and cache_read 0.10x lines (not just full-price)
- `dashboard.json` renders two panels: $/customer/day (dollar unit) and cache_hit_ratio (percentage unit, 0-1)
- Smoke test: hit-ratio for the second-onwards call shows the cache landing (> 0)
- Total spend under $3 (10 short Sonnet calls)


In [ ]:
# cost_exporter.py - the Ex 5 module (worked solution)
import anthropic
from prometheus_client import Counter, Gauge, start_http_server

COST = Counter("gen_ai_cost_usd_total", "Cumulative dollars spent on LLM calls",
               ["customer", "tier", "model"])
HIT  = Gauge("gen_ai_cache_hit_ratio", "Cache read tokens / total prefix tokens (0-1)",
             ["customer", "tier", "model"])

RATE_IN  = 2.0 / 1_000_000
RATE_OUT = 10.0 / 1_000_000
RATE_CW  = RATE_IN * 1.25   # cache write = 1.25x input
RATE_CR  = RATE_IN * 0.10   # cache read  = 0.10x input

def _cost(u) -> float:
    cw = getattr(u, "cache_creation_input_tokens", 0) or 0
    cr = getattr(u, "cache_read_input_tokens", 0) or 0
    return u.input_tokens * RATE_IN + cw * RATE_CW + cr * RATE_CR + u.output_tokens * RATE_OUT

def _hit_ratio(u) -> float:
    cw = getattr(u, "cache_creation_input_tokens", 0) or 0
    cr = getattr(u, "cache_read_input_tokens", 0) or 0
    total_prefix = cw + cr
    return (cr / total_prefix) if total_prefix > 0 else 0.0

class TrackedAnthropic:
    def __init__(self, customer: str, tier: str = "standard",
                 model: str = "claude-sonnet-5"):
        self.customer, self.tier, self.model = customer, tier, model
        self.client = anthropic.Anthropic()

    def call(self, system, user_q: str, max_tokens: int = 300):
        r = self.client.messages.create(model=self.model, max_tokens=max_tokens,
                                        system=system,
                                        messages=[{"role": "user", "content": user_q}])
        labels = dict(customer=self.customer, tier=self.tier, model=self.model)
        COST.labels(**labels).inc(_cost(r.usage))
        HIT.labels(**labels).set(_hit_ratio(r.usage))
        return r

# Production entrypoint - do NOT run inside a notebook (start_http_server blocks the kernel):
#   if __name__ == "__main__":
#       start_http_server(9100)         # GET http://localhost:9100/metrics
#       ...                             # warm two tenants, then keep the process alive

# Notebook smoke test - increment the metrics for two tenants and read the samples back
# without starting the HTTP server. Keyless mock returns zero-usage, so cost/hit are 0.0;
# a live key shows acmecorp/stripe diverging and hit_ratio climbing above 0 after call 1.
sys_block = [{"type": "text", "text": "Lending advisor preamble. " * 60,
              "cache_control": {"type": "ephemeral"}}]
acme = TrackedAnthropic(customer="acmecorp", tier="enterprise")
stp  = TrackedAnthropic(customer="stripe", tier="standard")
for _ in range(5):
    acme.call(sys_block, "Mi credit score es 700, eligible?")
    stp.call(sys_block, "Refund eligibility check")

for metric in list(COST.collect()) + list(HIT.collect()):
    for s in metric.samples:
        if s.name.endswith("_total") or s.name == "gen_ai_cache_hit_ratio":
            print(f"{s.name}{dict(s.labels)} = {s.value:.4f}")


In [ ]:
# dashboard.json - minimal Grafana panel spec (worked solution, Ex 5)
import json

DASHBOARD = {
    "title": "FDE Cost & Cache - per-customer per-day",
    "panels": [
        {
            "type": "timeseries",
            "title": "$/ customer / day",
            "targets": [
                {"expr": "sum by (customer) (increase(gen_ai_cost_usd_total[1d]))",
                 "legendFormat": "{{customer}}"}
            ],
            "fieldConfig": {"defaults": {"unit": "currencyUSD"}},
        },
        {
            "type": "timeseries",
            "title": "Cache hit ratio (0-1)",
            "targets": [
                {"expr": "avg by (customer) (gen_ai_cache_hit_ratio)",
                 "legendFormat": "{{customer}}"}
            ],
            "fieldConfig": {"defaults": {"unit": "percentunit", "min": 0, "max": 1}},
        },
    ],
    # Alert rule the productionized Ex 6 wires in: page when the prefix drifts.
    "alerts": [
        {"name": "cache_hit_ratio_low",
         "expr": "avg by (customer) (gen_ai_cache_hit_ratio) < 0.7",
         "for": "5m"}
    ],
}

with open("dashboard.json", "w", encoding="utf-8") as fh:
    json.dump(DASHBOARD, fh, indent=2)
print("wrote dashboard.json:", len(DASHBOARD["panels"]), "panels +",
      len(DASHBOARD["alerts"]), "alert rule")


---
## Exercise 6 - Graded: cache + batch a Stripe RAG agent, measure $/day, write the CFO memo

**Task.** Ship four files in `solutions/practice-3.5/ex6/`: (1) `rag_agent.py` - a Stripe-style merchant-support RAG agent whose ~3K-token system block (role + policy + few-shots) is cached; realtime path uses the cached prompt, overnight bulk enrichment (200 queries) goes through the batch API. (2) `measure.py` - run 100 representative queries through both paths and print a 4-row table: baseline (no cache, realtime), cached realtime, batch overnight, blended (70% realtime + 30% batch), scaled to 50K calls/day. (3) `dashboard/` - the Ex 5 `cost_exporter.py` + `dashboard.json`, productionized with tenant isolation and a `cache_hit_ratio < 0.7` alert rule. (4) `cfo_memo.md` - ~400 words leading with the headline $/month saving, assumptions, risks, and the runbook reference. Bonus (+1): `runbook.md` cache-invalidation runbook. The worked `measure.py` below is the graded harness; the other three files follow the same patterns.

**Success criteria**
- `rag_agent.py` runs end to end; realtime path uses cached prefix; batch path submits via the batch API
- `measure.py` prints the 4-row table; numbers come from real calls (sample of 20, not invented)
- `dashboard/cost_exporter.py` + `dashboard.json` reproduce Ex 5's panels with tenant labels
- `cfo_memo.md` is <= 1 page; headline number is in the first paragraph; assumptions + risks + runbook reference all present
- All numbers hedged ("approximate", "Aug 2026 rate sheet") - no false precision
- Total spend under $10 (the lab's $25 cap absorbs all 6 exercises with margin)


In [ ]:
# measure.py - the $/day comparison harness (worked solution, Ex 6)
# Ship alongside: rag_agent.py (cached realtime + batch overnight paths),
# dashboard/ (Ex 5 exporter + dashboard.json + the <0.7 alert rule), cfo_memo.md.
import json, anthropic
client = anthropic.Anthropic()

VOL_PER_DAY = 50_000
RATE_IN, RATE_OUT = 2.0/1_000_000, 10.0/1_000_000
RATE_CW, RATE_CR  = RATE_IN * 1.25, RATE_IN * 0.10
BATCH_DISCOUNT = 0.50

# The cached ~3K-token stable prefix (role + policy + few-shots) and its plain twin.
SYSTEM_BIG   = ("Stripe merchant-support RAG agent. OCC / FCA on payments + Stripe "
                "product docs + GDPR excerpts. code-mixed fluent. " * 60).strip()
SYSTEM_PLAIN = "Stripe merchant-support agent. Answer in 2-3 sentences."

def cost_one(u) -> float:
    cw = getattr(u, "cache_creation_input_tokens", 0) or 0
    cr = getattr(u, "cache_read_input_tokens", 0) or 0
    return u.input_tokens*RATE_IN + cw*RATE_CW + cr*RATE_CR + u.output_tokens*RATE_OUT

# Build a 100-line eval set (the customer's representative code-mixed queries).
SEED = [
    "Procesa el refund de pay_TEST29Qw, 18 dolares porfa",
    "El settlement no llego ni despues de T+2",
    "Por que el KYC esta pending todavia?",
    "El payment fallo pero me cobraron la card",
    "Mi card tiene 4 unauthorized transactions, es fraud",
]
with open("eval_100.jsonl", "w", encoding="utf-8") as fh:
    for i in range(100):
        row = {"query": SEED[i % 5] + (f" (variant {i})" if i >= 5 else "")}
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")

def baseline_call(q):
    r = client.messages.create(model="claude-sonnet-5", max_tokens=200,
                               system=SYSTEM_PLAIN,
                               messages=[{"role":"user","content":q}])
    return cost_one(r.usage)

def cached_call(q):
    r = client.messages.create(model="claude-sonnet-5", max_tokens=200,
                               system=[{"type":"text","text":SYSTEM_BIG,
                                        "cache_control":{"type":"ephemeral"}}],
                               messages=[{"role":"user","content":q}])
    return cost_one(r.usage)

QUERIES = [json.loads(l)["query"] for l in open("eval_100.jsonl", encoding="utf-8")]

baseline_avg = sum(baseline_call(q) for q in QUERIES[:20]) / 20  # sample 20 for the cost cap
cached_avg   = sum(cached_call(q)   for q in QUERIES[:20]) / 20
batch_avg    = cached_avg * BATCH_DISCOUNT  # batch + cache approximation
blend_avg    = 0.7 * cached_avg + 0.3 * batch_avg

print(f"{'mode':24s} | $/call | $/day @ 50K | $/month")
print("-" * 64)
for name, avg in [("baseline (no cache, RT)", baseline_avg),
                  ("cached realtime",          cached_avg),
                  ("batch overnight",          batch_avg),
                  ("blended 70/30",            blend_avg)]:
    daily = avg * VOL_PER_DAY
    print(f"{name:24s} | {avg:6.4f} | {daily:11.0f} | {daily*30:10.0f}")
print("\nKeyless mock returns zero-usage, so every row is $0.00. A live key on the sample-of-20 "
      "yields baseline > cached (60-80% off the prefix), batch = 0.5x cached, blend in between. "
      "All numbers are approximate and pinned to the Aug 2026 Sonnet 5 rate sheet - re-check on call day.")


---
## Exercise 6 - grading rubric + submission checklist

**Grading rubric (10 points, +1 bonus)**

| Axis | Weight | What full marks looks like |
|---|---|---|
| Correctness of cache placement | 3 | Cache breakpoint on the LAST stable block. Stable content (role, policy, few-shots) above it; per-call content (query, chunks, timestamp) below it. `cache_creation` fires once per session; `cache_read` fires on every subsequent call. Stable-block SHA-256 asserted identical across the 100-query run. |
| Measurable savings | 3 | 4-row table (baseline / cached / batch / blended $/day). Cached realtime shows 60-80% saving on the prefix; batch shows 50% off cached realtime; blended matches the 70/30 weight. Numbers hedged with an "Aug 2026 rate sheet" caveat. |
| Cost dashboard | 2 | `cost_exporter.py` wired into `rag_agent.py`, metrics labelled by customer + tier + model. `dashboard.json` renders both panels. Alert rule (`cache_hit_ratio < 0.7 for 5m`) defined. Tenant isolation enforced. |
| CFO-readable memo | 2 | `cfo_memo.md` is ~400 words, leads with the headline $/month saving, names the assumption set (volume, prefix length, rate date), names two risks (cache invalidation on policy revisions, batch SLA on tight-deadline workloads), references the runbook. No jargon a CFO would not recognize. |
| Bonus: cache-invalidation runbook | +1 | `runbook.md`: 4 numbered steps (compute new prefix hash, warm with canary, verify hit-ratio recovery, fall back + page on-call). References the alert rule. Half a page or less. |

**Submission checklist**
- [ ] `rag_agent.py` runs end to end; realtime path uses cached prefix; batch path submits via the batch API
- [ ] `measure.py` prints the 4-row table; numbers come from real calls (sample of 20, not invented)
- [ ] `dashboard/cost_exporter.py` + `dashboard.json` reproduce Ex 5's panels with tenant labels
- [ ] `cfo_memo.md` <= 1 page; headline number in the first paragraph; assumptions + risks + runbook reference present
- [ ] All numbers hedged ("approximate", "Aug 2026 rate sheet") - no false precision
- [ ] Total spend under $10 (the lab's $25 cap absorbs all 6 exercises with margin)

**Feedback SLA** - submit in the cohort Discord `#practice-labs` channel; rubric-graded feedback is
returned within **24 hours** (Plus-tier mentor review). Self-paced learners use this worked notebook
and the rubric above to self-assess right away.
